In [1]:
from pyspark.sql import SparkSession
import pyspark

AWS_ACCESS_KEY = "minioadmin"
AWS_SECRET_KEY = "minioadmin"
AWS_S3_ENDPOINT = "http://minio_server:9000"
WAREHOUSE = "s3a://gold/" 
NESSIE_URI = "http://nessie:19120/api/v1"

conf = (
    pyspark.SparkConf()
    .setAppName("TrainModel2")  
    .set('spark.jars.packages',
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'org.apache.hadoop:hadoop-aws:3.3.4,'
         'com.amazonaws:aws-java-sdk-bundle:1.12.300')
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.authentication.type", "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint", AWS_S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.access-key", AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-key", AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
)

spark = (
    SparkSession.builder
    .config(conf=conf) 
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")


In [2]:
df_fact = spark.table("nessie.gold_db.fact_order")
df_customer = spark.table("nessie.gold_db.dim_customer")
df_product = spark.table("nessie.gold_db.dim_product")
df_time = spark.table("nessie.gold_db.dim_time")
df_location = spark.table("nessie.gold_db.dim_location")

In [3]:
query = """
SELECT 
    f.time_id,
    f.customer_id,
    f.product_id,
    f.location_id,
    f.quantity,
    f.unit_price,        
    f.total_revenue,     

    -- Dim_time
    t.order_date,
    t.year,
    t.month,
    t.day,
    t.quarter,
    t.weekday_name,

    -- Dim_customer
    c.gender,
    c.age_group,
    c.education,
    c.income,
    c.race,
    c.state,             -- Cột state từ bảng khách hàng

    -- Dim_product
    p.product_title,
    p.product_category,

    -- Dim_location
    l.state_name purchase_state,
    l.region purchase_region

FROM nessie.gold_db.fact_order f
LEFT JOIN nessie.gold_db.dim_time t     ON f.time_id     = t.time_id
LEFT JOIN nessie.gold_db.dim_customer c ON f.customer_id  = c.customer_id
LEFT JOIN nessie.gold_db.dim_product p   ON f.product_id  = p.product_id
LEFT JOIN nessie.gold_db.dim_location l  ON f.location_id = l.location_id
"""

In [4]:
df_fact_full = spark.sql(query)
df_fact_full.limit(10).toPandas()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

,time_id,customer_id,product_id,location_id,quantity,unit_price,total_revenue,order_date,year,month,...,gender,age_group,education,income,race,state,product_title,product_category,purchase_state,purchase_region
0,20221126,R_262vJPpUeeFKuHT,0307118932,34359738369,1.0,3.99,3.99,2022-11-26,2022,11,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Just Grandma and Me (Little Critter) (Pictureb...,ABIS_BOOK,Arizona,West
1,20210419,R_2AWkL1j4mAcAll8,0307464970,34359738369,1.0,15.99,15.99,2021-04-19,2021,4,...,Male,18 - 24 years,High school diploma or GED,"$25,000 - $49,999",American Indian/Native American or Alaska Native,Arizona,The Harlem Hellfighters,ABIS_BOOK,Arizona,West
2,20200609,R_tMsVLA8C6RuTmlb,048640708X,34359738369,1.0,7.89,7.89,2020-06-09,2020,6,...,Female,55 - 64 years,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Chinese Painting Techniques (Dover Art Instruc...,ABIS_BOOK,Arizona,West
3,20191221,R_262vJPpUeeFKuHT,0812971833,34359738369,1.0,12.60,12.60,2019-12-21,2019,12,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Olive Kitteridge,ABIS_BOOK,Arizona,West
4,20221214,R_AmMCFCEBSSiRhLz,1492677698,34359738369,1.0,17.99,17.99,2022-12-14,2022,12,...,Female,35 - 44 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Arizona,The Complete Baking Book for Young Chefs: 100+...,ABIS_BOOK,Arizona,West
5,20200826,R_3G9b1ToPNT28Utq,1627797645,34359738369,1.0,6.50,6.50,2020-08-26,2020,8,...,Female,25 - 34 years,High school diploma or GED,"$50,000 - $74,999",White or Caucasian,Arizona,Marlena: A Novel,ABIS_BOOK,Arizona,West
6,20221004,R_262vJPpUeeFKuHT,1637315309,34359738369,1.0,11.99,11.99,2022-10-04,2022,10,...,Female,65 and older,"Graduate or professional degree (MA, MS, MBA, ...","$25,000 - $49,999",White or Caucasian,Arizona,Hooty Pooty the Owl: A Funny Rhyming Halloween...,ABIS_BOOK,Arizona,West
7,20190401,R_2E0Q3pbtzJQqLg7,B00004R9TL,34359738369,1.0,20.85,20.85,2019-04-01,2019,4,...,Male,35 - 44 years,Bachelor's degree,"$25,000 - $49,999",White or Caucasian,Florida,"BLACK+DECKER Trimmer Line, 30-Foot, 0.065-Inch...",IGNITION_COIL,Arizona,West
8,20191127,R_06RZP9pS7kONINr,B00006ANDK,996432412672,1.0,19.99,19.99,2019-11-27,2019,11,...,Female,65 and older,Bachelor's degree,"$75,000 - $99,999",White or Caucasian,South Dakota,Oral-B Sensitive Gum Care Electric Toothbrush ...,TOOTHBRUSH_HEAD,South Dakota,Midwest
9,20210118,R_4PC8xUs7y9n494R,B0000BYEF6,34359738369,2.0,12.34,24.68,2021-01-18,2021,1,...,Female,25 - 34 years,Bachelor's degree,"$100,000 - $149,999",White or Caucasian,Arizona,Lutron Credenza Plug-In Dimmer for Incandescen...,ELECTRONIC_SWITCH,Arizona,West


In [5]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# 1. TÌM NGÀY CUỐI CÙNG TRONG DỮ LIỆU ĐỂ TÍNH RECENCY
# Lấy giá trị ngày lớn nhất (mới nhất) trong cột order_date để làm mốc tính độ tươi mới (Recency)
max_date_val = df_fact_full.select(F.max("order_date")).collect()[0][0]

# =====================================================
# 1. FEATURE ENGINEERING 
# =====================================================
# Gom nhóm dữ liệu theo từng khách hàng để xây dựng bộ tính năng đặc trưng (Features)
df_enhanced = (
    df_fact_full.groupBy("customer_id")
    .agg(
        F.sum("total_revenue").alias("total_spend"),              # Tổng chi tiêu của khách hàng
        F.countDistinct("time_id").alias("total_orders"),         # Tổng số đơn hàng duy nhất
        F.countDistinct("product_category").alias("category_diversity"), # Số lượng danh mục sản phẩm đã mua
        F.avg("quantity").alias("avg_items_per_line"),            # Số lượng sản phẩm trung bình mỗi dòng đơn hàng
        F.min("year").alias("first_year"),                        # Năm đầu tiên bắt đầu mua hàng
        F.max("year").alias("last_year"),                         # Năm cuối cùng phát sinh giao dịch
        
        # --- BIẾN MỚI 1: Recency (Độ tươi mới) ---
        # Tính số ngày từ lần mua cuối cùng đến ngày hiện tại (mốc max_date)
        F.datediff(F.lit(max_date_val), F.max("order_date")).alias("recency"),
        
        # Lấy thuộc tính nhóm tuổi (nhân khẩu học) của khách hàng
        F.first("age_group").alias("age_group")
    )
    # Tính số năm khách hàng đã hoạt động tại hệ thống
    .withColumn("years_active", (F.col("last_year") - F.col("first_year") + 1))
    # Đảm bảo số năm hoạt động tối thiểu là 1 để tránh lỗi chia cho 0
    .withColumn("years_active", F.when(F.col("years_active") <= 0, 1).otherwise(F.col("years_active")))
    
    # --- TARGET: Biến mục tiêu cần dự đoán ---
    # Tần suất đơn hàng trung bình mỗi năm (Số đơn hàng / Số năm hoạt động)
    .withColumn("avg_orders_per_year", F.col("total_orders") / F.col("years_active"))
    
    # --- BIẾN MỚI 2: Giá trị trung bình của mỗi đơn hàng (AOV) ---
    .withColumn("avg_order_value", F.col("total_spend") / F.col("total_orders"))
    
    # --- Xử lý phân phối dữ liệu (Log Transform) ---
    # Sử dụng log1p (log(1+x)) để nén dải giá trị rộng, giảm tác động của các giá trị ngoại lai (Outliers)
    .withColumn("total_spend_log", F.log1p("total_spend"))
    .withColumn("avg_order_value_log", F.log1p("avg_order_value"))
    .withColumn("recency_log", F.log1p("recency"))
    .withColumn("avg_items_log", F.log1p("avg_items_per_line"))
    
    # Xử lý giá trị thiếu (Null values)
    .na.fill("Unknown", ["age_group"])
    .na.fill(0)
)

# =====================================================
# 2. TIỀN XỬ LÝ (PRE-PROCESSING PIPELINE)
# =====================================================
# Định nghĩa các nhóm cột đầu vào cho mô hình
cat_cols = ["age_group"] # Cột định tính (Categorical)
num_cols = ["total_spend_log", "category_diversity", "avg_order_value_log", "recency_log", "years_active"] # Cột định lượng (Numerical)

# Xử lý biến định tính: Chuyển đổi chữ (String) -> Chỉ số (Index) -> Vector nhị phân (OneHot)
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") for c in cat_cols]

# Gom tất cả các tính năng (số và vector định tính) vào một cột vector duy nhất là 'rawFeatures'
assembler = VectorAssembler(
    inputCols=num_cols + [f"{c}_vec" for c in cat_cols],
    outputCol="rawFeatures"
)

# Chuẩn hóa dữ liệu (Standardization): Đưa các biến về cùng thang đo (Mean=0, Std=1)
# Giúp mô hình hội tụ nhanh hơn và hoạt động ổn định hơn
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withMean=True, withStd=True)

RandomForest

In [9]:
from pyspark.ml.regression import RandomForestRegressor
import numpy as np
# =====================================================
# 3. MÔ HÌNH VÀ TUNING
# =====================================================
# Định nghĩa Model
rf = RandomForestRegressor(
    labelCol="avg_orders_per_year", 
    featuresCol="features",
    seed=42
)

# Xây dựng Pipeline tổng thể
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

# Chia dữ liệu Train/Test
train_df, test_df = df_enhanced.randomSplit([0.8, 0.2], seed=42)

# Thiết lập Grid tìm tham số tốt nhất
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [200, 300 ,400])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)

evaluator = RegressionEvaluator(labelCol="avg_orders_per_year", metricName="r2")

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=4
)
# --------------------------
# 4. HUẤN LUYỆN
# --------------------------
print("Đang huấn luyện mô hình và tìm kiếm tham số tối ưu...")
rf_tvs_model = tvs.fit(train_df)

# --------------------------
# 5. TRÍCH XUẤT VÀ IN THAM SỐ TỐI ƯU
# --------------------------
# Lấy stage cuối cùng của Pipeline tốt nhất (chính là RandomForestRegressor)
best_pipeline_model = rf_tvs_model.bestModel
best_rf = best_pipeline_model.stages[-1]

print("\n" + "="*30)
print("THAM SỐ TỐI ƯU (BEST PARAMETERS):")
print("="*30)

# In các tham số cụ thể đã được Tuning
print(f"Số lượng cây (numTrees) : {best_rf.getNumTrees}")
print(f"Độ sâu tối đa (maxDepth): {best_rf.getOrDefault('maxDepth')}")

Đang huấn luyện mô hình và tìm kiếm tham số tối ưu...

THAM SỐ TỐI ƯU (BEST PARAMETERS):
Số lượng cây (numTrees) : 400
Độ sâu tối đa (maxDepth): 8


In [15]:
# ĐÁNH GIÁ
# Sử dụng bestModel để dự đoán trên cả 2 tập dữ liệu
train_pred = rf_tvs_model.bestModel.transform(train_df)
test_pred  = rf_tvs_model.bestModel.transform(test_df)

# Thiết lập target (đã định nghĩa ở bước trước là avg_orders_per_year)
target = "avg_orders_per_year"

print("\n" + "="*30)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH")
print("="*30)

# --------------------------
# Vòng lặp tính toán các chỉ số (Metrics)
# --------------------------
metrics = ['r2', 'mae', 'rmse']

for metric in metrics:
    # Khởi tạo evaluator cho từng chỉ số
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    
    # Tính toán kết quả cho tập Train và Test
    train_score = evaluator.evaluate(train_pred)
    test_score  = evaluator.evaluate(test_pred)
    
    # In kết quả 
    print(f"{metric.upper()} Train:", round(train_score, 4))
    print(f"{metric.upper()} Test :", round(test_score, 4))


KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH
R2 Train: 0.9594
R2 Test : 0.9154
MAE Train: 3.3454
MAE Test : 4.2076
RMSE Train: 5.6306
RMSE Test : 8.8144


In [11]:
import mlflow
import mlflow.spark
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Cấu hình MLflow
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Order_Frequency_Prediction") 

# 2. Chuẩn bị Target và Metrics 
target = "avg_orders_per_year"
metrics_list = ['r2', 'mae', 'rmse']
metrics_dict = {}

# Tính toán và lưu metrics vào dictionary để log một lượt
for metric in metrics_list:
    evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    metrics_dict[f"{metric}_train"] = round(evaluator.evaluate(train_pred), 4)
    metrics_dict[f"{metric}_test"]  = round(evaluator.evaluate(test_pred), 4)

# 3. Lấy thông tin model tốt nhất từ Pipeline
best_rf = rf_tvs_model.bestModel.stages[-1]

# 4. Ghi dữ liệu vào MLflow
with mlflow.start_run(run_name="RandomForest_Order_Frequency"):
    
    # --- LOG PARAMS (Tham số cấu hình) ---
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", best_rf.getNumTrees)
    mlflow.log_param("maxDepth", best_rf.getOrDefault("maxDepth"))
    mlflow.log_param("seed", 42)
    
    # --- LOG METRICS (Kết quả đánh giá) ---
    # Vòng lặp log toàn bộ r2, mae, rmse cho cả train và test
    for k, v in metrics_dict.items():
        mlflow.log_metric(k, v)
    
    # --- LOG MODEL (Lưu file model để tái sử dụng) ---
    # Lưu toàn bộ PipelineModel (bao gồm cả các bước xử lý StringIndexer, VectorAssembler...)
    mlflow.spark.log_model(rf_tvs_model.bestModel, "random_forest_order_model")

    print("\n✅ [MLflow] Đã lưu thành công phiên bản mô hình dự đoán số đơn hàng!")
    print(f"📈 R2 Test hiện tại: {metrics_dict['r2_test']}")

2026/04/21 04:49:55 INFO mlflow.tracking.fluent: Experiment with name 'Customer_Order_Frequency_Prediction' does not exist. Creating a new experiment.



✅ [MLflow] Đã lưu thành công phiên bản mô hình dự đoán số đơn hàng!
📈 R2 Test hiện tại: 0.9154
🏃 View run RandomForest_Order_Frequency at: http://mlflow:5000/#/experiments/3/runs/166ddde826c04e9eaf6e8a9f9e7c863b
🧪 View experiment at: http://mlflow:5000/#/experiments/3


XGB

In [13]:
# =====================================================
# 3. MÔ HÌNH XGBOOST
# =====================================================
target = "avg_orders_per_year"

xgb = SparkXGBRegressor(
    features_col="features",
    label_col=target,
    num_workers=2,
    tree_method="hist",
    # Đổi sang Tweedie vì target là số lượng đơn hàng (count data)
    objective="reg:tweedie",
    tweedie_variance_power=1.5, 
    seed=42
)
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, xgb])

train_df, test_df = df_enhanced.randomSplit([0.8, 0.2], seed=42)

param_grid = (
    ParamGridBuilder()
    .addGrid(xgb.max_depth, [3, 5, 7])
    .addGrid(xgb.learning_rate, [0.02, 0.05])
    .addGrid(xgb.reg_lambda, [100.0, 200.0])
    .addGrid(xgb.n_estimators, [350, 550])
    .addGrid(xgb.subsample, [0.7, 0.8])
    .build()
)

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=RegressionEvaluator(labelCol=target, metricName="r2"),
    trainRatio=0.8,
    parallelism=4,
    seed=42
)

print("Đang huấn luyện XGBoost ")
xgb_tvs_model = tvs.fit(train_df)

# --------------------------
# TRÍCH XUẤT VÀ IN THAM SỐ TỐI ƯU XGBOOST
# --------------------------

# 1. Tìm chỉ số (index) của bộ tham số có R2 cao nhất
best_index = np.argmax(xgb_tvs_model.validationMetrics)
# Lấy map chứa các tham số tương ứng với index đó
best_params = xgb_tvs_model.getEstimatorParamMaps()[best_index]

print(f"\nBest XGBoost Parameters (from grid search):")

# 2. Duyệt qua bộ tham số và in theo định dạng yêu cầu
for param, value in best_params.items():
    # param.name sẽ lấy tên tham số (ví dụ: max_depth)
    print(f" - {param.name}: {value}")

Đang huấn luyện XGBoost 


2026-04-21 05:36:21,309 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 2 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.02, 'max_depth': 3, 'objective': 'reg:tweedie', 'reg_lambda': 100.0, 'subsample': 0.8, 'tree_method': 'hist', 'tweedie_variance_power': 1.5, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 350}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-04-21 05:36:21,309 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 2 workers with
	booster params: {'device': 'cpu', 'learning_rate': 0.02, 'max_depth': 3, 'objective': 'reg:tweedie', 'reg_lambda': 100.0, 'subsample': 0.8, 'tree_method': 'hist', 'tweedie_variance_power': 1.5, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 350}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-04-21 05:36:21,340 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 2 workers with
	booster params: {'device': 'cpu', 'learn


Best XGBoost Parameters (from grid search):
 - max_depth: 5
 - learning_rate: 0.05
 - reg_lambda: 100.0
 - n_estimators: 550
 - subsample: 0.7


In [16]:
# =====================================================
# 4. ĐÁNH GIÁ (R2, MAE, RMSE)
# =====================================================
# Sử dụng mô hình tốt nhất để dự đoán trên cả 2 tập dữ liệu
train_pred = xgb_tvs_model.bestModel.transform(train_df)
test_pred  = xgb_tvs_model.bestModel.transform(test_df)

print("\n KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH :")
print("-" * 35)

for metric in ['r2', 'mae', 'rmse']:
    eval_m = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    
    # Tính toán giá trị cho tập Train và Test
    train_score = eval_m.evaluate(train_pred)
    test_score  = eval_m.evaluate(test_pred)
    
    # In kết quả 
    print(f"{metric.upper():<4} Train: {round(train_score, 4)}")
    print(f"{metric.upper():<4} Test : {round(test_score, 4)}")


 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH :
-----------------------------------
R2   Train: 0.9978
R2   Test : 0.9807
MAE  Train: 0.7227
MAE  Test : 1.3586
RMSE Train: 1.3154
RMSE Test : 4.2082


In [18]:
import mlflow
import mlflow.spark
import numpy as np

# 1. Cấu hình MLflow
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Order_Frequency_Prediction")

# 2. Chuẩn bị Dictionary để chứa toàn bộ Metrics (Khớp với vòng lặp của bạn)
metrics_to_log = {}
for metric in ['r2', 'mae', 'rmse']:
    eval_m = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName=metric)
    metrics_to_log[f"{metric}_train"] = round(eval_m.evaluate(train_pred), 4)
    metrics_to_log[f"{metric}_test"]  = round(eval_m.evaluate(test_pred), 4)

# 3. Trích xuất tham số tối ưu từ Grid Search
best_index = np.argmax(xgb_tvs_model.validationMetrics)
best_params_map = xgb_tvs_model.getEstimatorParamMaps()[best_index]
best_xgb_model = xgb_tvs_model.bestModel.stages[-1]

# 4. Ghi dữ liệu vào MLflow
with mlflow.start_run(run_name="XGBoost_Tweedie_Final"):
    
    # --- LOG PARAMS ---
    # Log thủ công các tham số quan trọng từ Grid Search
    for param, value in best_params_map.items():
        mlflow.log_param(param.name, value)
    
    # Log thêm cấu hình đặc thù của mô hình
    mlflow.log_param("objective", "reg:tweedie")
    mlflow.log_param("tweedie_variance_power", 1.5)
    
    # --- LOG METRICS ---
    # Log toàn bộ R2, MAE, RMSE cho cả Train và Test
    for metric_name, metric_value in metrics_to_log.items():
        mlflow.log_metric(metric_name, metric_value)
    
    # --- LOG MODEL ---
    # Lưu toàn bộ PipelineModel (bao gồm các bước xử lý dữ liệu và model XGB)
    mlflow.spark.log_model(xgb_tvs_model.bestModel, "xgb_order_frequency_model")

    print("\n✅ [MLflow] Đã đóng gói và lưu mô hình XGBoost thành công!")
    print(f"📊 R2 Test logged: {metrics_to_log['r2_test']}")


✅ [MLflow] Đã đóng gói và lưu mô hình XGBoost thành công!
📊 R2 Test logged: 0.9807
🏃 View run XGBoost_Tweedie_Final at: http://mlflow:5000/#/experiments/3/runs/e5e5840ac8ea4188ac0d48da2a46253d
🧪 View experiment at: http://mlflow:5000/#/experiments/3
